In [1]:
import os
import cv2
import json
import numpy as np
from tqdm import tqdm
import shutil
from skimage.measure import label

In [2]:
out='/kaggle/working/filed_data'
os.makedirs(os.path.join(out, 'images'), exist_ok=True)

In [3]:
cls={5: "Grasper",
    9: "L-hook",
    13: "Trocar",
    14: "Scissor",
    15: "Clipper",
    16: "Irrigator",
    17: "SpecimenBag"}

In [4]:
vfols=sorted([f for f in os.listdir('/kaggle/input/cholecseg8k')])
idxs=43

In [5]:
len(vfols)

17

In [6]:
ctrain={'images':[],
        'annotations':[],
        'categories':[{'id':k,'name':v,'supercategory':'tools'}for k,v in cls.items()]
}
cval={'images':[],
        'annotations':[],
        'categories':[{'id':k,'name':v,'supercategory':'tools'}for k,v in cls.items()]
}

In [7]:
temap=[]

In [8]:
aid,gimidx=1,1

In [9]:
for vfol in tqdm(vfols):
    it=vfols.index(vfol)<idxs
    cococ=ctrain if it else cval
    vpath=os.path.join('/kaggle/input/cholecseg8k',vfol)
    ffols=sorted([f for f in os.listdir(vpath)])
    pimgf=None
    for ffol in ffols:
        fpath=os.path.join(vpath,ffol)
        rawimgs=sorted([f for f in os.listdir(fpath) if f.endswith('_endo.png')])
        for rawimg in rawimgs:
            pre=rawimg.replace('.png','')
            mskfil=f'{pre}_mask.png'
            imgpth=os.path.join(fpath,rawimg)
            mskpth=os.path.join(fpath,mskfil)
            img=cv2.imread(imgpth)
            mskimg=cv2.imread(mskpth,cv2.IMREAD_GRAYSCALE)
            h,w=img.shape[:2]
            nfile=f'{vfol}_{ffol}_{rawimg}'
            onpth=os.path.join(out,'images',nfile)
            shutil.copy(imgpth, onpth)
            cococ['images'].append({
            'id':gimidx,
            'file_name':nfile,
            'height':h,
            'width':w,
            'video_id':vfol
            })
            for clsidx,clsn in cls.items():
                clsmsk=(mskimg==clsidx).astype(np.uint8)
                lblmsk,numinst=label(clsmsk,return_num=True)
                for inst in range(1,numinst+1):
                    insmsk=(lblmsk==inst).astype(np.uint8)
                    cntr,_=cv2.findContours(insmsk,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
                    plyg=[c.flatten().tolist() for c in cntr if len(c)>=3]
                    if not plyg:continue
                    pos=np.where(insmsk)
                    xm,xma=np.min(pos[1]),np.max(pos[1])
                    ym,yma=np.min(pos[0]),np.max(pos[0])
                    bbox=[int(xm),int(ym),int(xma-xm),int(yma-ym)]
                    cococ['annotations'].append({
                    'id':aid,
                    'image_id':gimidx,
                    'category_id':clsidx,
                    'segmentation':plyg,
                    'area':int(np.sum(insmsk)),
                    'bbox':bbox,
                    'iscrowd':0
                })
                    aid+=1
            if pimgf is not None:
                temap.append({'current':nfile,'previous':pimgf,'video':vfol})
            pimgf=nfile
            gimidx+=1
    print('video')

  6%|▌         | 1/17 [01:56<30:56, 116.06s/it]

video


 12%|█▏        | 2/17 [02:19<15:21, 61.40s/it] 

video


 18%|█▊        | 3/17 [03:16<13:51, 59.37s/it]

...

video


 94%|█████████▍| 16/17 [12:06<00:45, 45.14s/it]

video


100%|██████████| 17/17 [12:26<00:00, 43.89s/it]

video


In [10]:
with open(os.path.join(out,'train_instances.json'),'w') as f:
    json.dump(ctrain,f)

In [11]:
with open(os.path.join(out,'val_stances.json'),'w') as f:
    json.dump(cval,f)

In [12]:
with open(os.path.join(out,'temporal.json'),'w') as f:
    json.dump(temap,f)

In [13]:
gimidx-1

8080